![](Images/banner_health_analytics.png){fig-align="center" width="100%"}

### Health Analytics with Python · Module 08: Reproducible Research & Deployment
***
**Learning objectives**
- Understand Quarto's role in reproducible health research reporting
- Create multi-format Quarto documents (HTML, PDF, Word)
- Build publication-quality figures with embedded captions and cross-references
- Generate programmatic Table 1 with statistical tests
- Apply Quarto callouts, tabs, and interactive elements
- Use Quarto for journal submission workflows

**Estimated time:** 2.5 hours | **Level:** Intermediate | **Libraries:** `matplotlib`, `pandas`, `sklearn`


## 1. Setup & Dataset

In [ ]:
import os, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
os.makedirs("/tmp/mod08", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white"})
np.random.seed(42); N = 2000
df = pd.DataFrame({
    "patient_id" : [f"PT-{i:05d}" for i in range(N)],
    "age"        : np.random.normal(60,15,N).clip(18,90).astype(int),
    "readmitted" : np.random.binomial(1, 0.18, N),
    "los_days"   : np.random.gamma(2.5,1.8,N).clip(1,30).astype(int),
    "diabetes"   : np.random.binomial(1, 0.35, N),
    "hf"         : np.random.binomial(1, 0.22, N),
    "smoking"    : np.random.binomial(1, 0.25, N),
    "payer"      : np.random.choice(["Medicare","Medicaid","Private","Self-pay"],N,p=[0.40,0.22,0.28,0.10]),
})
def sigmoid(x): return 1/(1+np.exp(-x))
logit = -3.2+0.6*df.hf+0.5*df.diabetes+0.02*(df.age-60)+0.2*(df.los_days>7).astype(int)
df["readmitted"] = np.random.binomial(1,sigmoid(logit.values),N)
print(f"Dataset: {df.shape} | Readmission rate: {df.readmitted.mean()*100:.1f}%")


## 2. Quarto Document Structure

A `.qmd` file is a Markdown file with YAML front matter and executable code chunks:

```
---                          <- YAML front matter (metadata)
title: "Report Title"
format: html
***
## Section                   <- Markdown content

```{python}                  <- Executable code chunk
#| label: fig-roc            <- Cross-referenceable label
#| fig-cap: "ROC Curve"      <- Caption
plot_roc_curve()
```

See @fig-roc for the discrimination results.   <- Auto-numbered cross-ref
```

**Key Quarto advantages over Jupyter:**
- Clean separation of code and output
- Native multi-format (HTML/PDF/Word) from same source
- Built-in bibliography (`.bib`) support
- Cross-references, callouts, panel layouts
- Version-control friendly (`.qmd` is plain text)


## 3. Quarto YAML Front Matter Template

In [ ]:
# The Quarto document template for health research reports
QMD_TEMPLATE = (
    "---\n"
    "title: '30-Day Readmission Analysis Report'\n"
    "subtitle: 'Health Analytics Division | Fiscal Year 2024'\n"
    "author:\n"
    "  - name: Dr. Data Analyst\n"
    "    affiliation: Health System Analytics\n"
    "    email: analyst@healthsystem.org\n"
    "date: today\n"
    "format:\n"
    "  html:\n"
    "    toc: true\n"
    "    toc-depth: 3\n"
    "    theme: cosmo\n"
    "    embed-resources: true\n"
    "    code-fold: true\n"
    "    code-summary: 'Show code'\n"
    "  pdf:\n"
    "    documentclass: article\n"
    "    geometry: margin=1in\n"
    "    toc: true\n"
    "  docx:\n"
    "    reference-doc: templates/report_template.docx\n"
    "execute:\n"
    "  echo: false\n"
    "  warning: false\n"
    "  cache: true\n"
    "bibliography: references.bib\n"
    "---\n\n"
    "## Executive Summary\n\n"
    "::: {.callout-note}\n"
    "**Key Finding:** 30-day readmission rate was {{< readmit_rate >}}%, with highest risk\n"
    "among patients with heart failure (OR {{< hf_or >}}, 95% CI {{< hf_ci >}}).\n"
    "::::\n\n"
    "## Methods\n\n"
    "```{python}\n"
    "#| label: tbl-patient-characteristics\n"
    "#| tbl-cap: 'Patient characteristics by readmission status'\n"
    "# Table 1 code here\n"
    "```\n\n"
    "## Results\n\n"
    "```{python}\n"
    "#| label: fig-roc-curve\n"
    "#| fig-cap: 'ROC curve for 30-day readmission model'\n"
    "#| fig-width: 8\n"
    "#| fig-height: 5\n"
    "# Figure code here\n"
    "```\n\n"
    "@fig-roc-curve shows model discrimination (AUC = 0.78, 95% CI: 0.74-0.81).\n\n"
    "## References\n\n"
    "::: {#refs}\n"
    ":::\n"
)
print("Quarto document template:")
print(QMD_TEMPLATE[:1200])
qmd_path = Path("/tmp/mod08/readmission_report.qmd")
qmd_path.write_text(QMD_TEMPLATE)
print(f"\nTemplate written to: {qmd_path}")


## 4. Publication-Quality Figures (300 dpi)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix
import seaborn as sns

X = df[["age","los_days","diabetes","hf","smoking"]]
y = df["readmitted"]
X_tr,X_te,y_tr,y_te = train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)
lr = LogisticRegression(C=1,max_iter=500,class_weight="balanced",random_state=42)
lr.fit(X_tr,y_tr)
prob = lr.predict_proba(X_te)[:,1]
auc  = roc_auc_score(y_te, prob)

fig = plt.figure(figsize=(18,10))
gs  = gridspec.GridSpec(2,3,figure=fig,hspace=0.38,wspace=0.35)

# Panel A — ROC
ax_a = fig.add_subplot(gs[0,0])
fpr,tpr,_ = roc_curve(y_te,prob)
ax_a.plot(fpr,tpr,color="#D65F5F",lw=2.5,label=f"AUC={auc:.3f}")
ax_a.plot([0,1],[0,1],"k--",lw=1)
ax_a.fill_between(fpr,tpr,alpha=0.1,color="#D65F5F")
ax_a.set_xlabel("1 - Specificity"); ax_a.set_ylabel("Sensitivity")
ax_a.set_title("A  ROC Curve",fontweight="bold"); ax_a.legend(fontsize=9)
ax_a.spines["top"].set_visible(False); ax_a.spines["right"].set_visible(False)

# Panel B — Score distribution
ax_b = fig.add_subplot(gs[0,1])
for val,color,lbl in [(0,"#4878CF","Not readmitted"),(1,"#D65F5F","Readmitted")]:
    ax_b.hist(prob[y_te==val],bins=25,alpha=0.65,color=color,label=lbl,density=True)
ax_b.set_xlabel("Predicted probability"); ax_b.set_ylabel("Density")
ax_b.set_title("B  Score Distribution",fontweight="bold"); ax_b.legend(fontsize=8)
ax_b.spines["top"].set_visible(False); ax_b.spines["right"].set_visible(False)

# Panel C — Table 1 (bar chart substitute)
ax_c = fig.add_subplot(gs[0,2])
features = ["HF","Diabetes","Smoking","LOS>7d","Age>70"]
readmit_rates = [
    df[df.hf==1]["readmitted"].mean()*100,
    df[df.diabetes==1]["readmitted"].mean()*100,
    df[df.smoking==1]["readmitted"].mean()*100,
    df[df.los_days>7]["readmitted"].mean()*100,
    df[df.age>70]["readmitted"].mean()*100,
]
baseline = df.readmitted.mean()*100
colors_bar = ["#D65F5F" if r > baseline+3 else "#4878CF" for r in readmit_rates]
ax_c.barh(features,readmit_rates,color=colors_bar,alpha=0.85,edgecolor="white")
ax_c.axvline(baseline,color="black",ls="--",lw=1.5,label=f"Overall ({baseline:.1f}%)")
ax_c.set_xlabel("30-day readmission rate (%)"); ax_c.set_title("C  Readmission by Factor",fontweight="bold")
ax_c.legend(fontsize=8); ax_c.spines["top"].set_visible(False); ax_c.spines["right"].set_visible(False)

# Panel D — Payer analysis
ax_d = fig.add_subplot(gs[1,0])
payer_rates = df.groupby("payer")["readmitted"].mean().sort_values(ascending=False)*100
colors_p = ["#D65F5F","#D4A017","#4878CF","#6ACC65"]
bars = ax_d.bar(payer_rates.index,payer_rates.values,color=colors_p,alpha=0.85,edgecolor="white")
ax_d.axhline(baseline,color="black",ls="--",lw=1.5)
ax_d.set_ylabel("Readmission rate (%)"); ax_d.set_title("D  Readmission by Payer",fontweight="bold")
ax_d.tick_params(axis="x",rotation=15)
ax_d.spines["top"].set_visible(False); ax_d.spines["right"].set_visible(False)
for bar,val in zip(bars,payer_rates.values):
    ax_d.text(bar.get_x()+bar.get_width()/2,val+0.3,f"{val:.1f}%",ha="center",fontsize=8)

# Panel E — Feature importance
ax_e = fig.add_subplot(gs[1,1])
coefs = pd.Series(lr.coef_[0],index=X.columns).sort_values()
colors_coef = ["#D65F5F" if v>0 else "#4878CF" for v in coefs.values]
ax_e.barh(coefs.index,coefs.values,color=colors_coef,alpha=0.85,edgecolor="white")
ax_e.axvline(0,color="black",lw=0.8)
ax_e.set_xlabel("Log-odds coefficient"); ax_e.set_title("E  Model Coefficients",fontweight="bold")
ax_e.spines["top"].set_visible(False); ax_e.spines["right"].set_visible(False)

# Panel F — LOS distribution
ax_f = fig.add_subplot(gs[1,2])
for val,color,lbl in [(0,"#4878CF","Not readmitted"),(1,"#D65F5F","Readmitted")]:
    ax_f.hist(df[df.readmitted==val]["los_days"],bins=20,alpha=0.6,color=color,label=lbl,density=True)
ax_f.set_xlabel("Length of Stay (days)"); ax_f.set_ylabel("Density")
ax_f.set_title("F  LOS Distribution",fontweight="bold"); ax_f.legend(fontsize=8)
ax_f.spines["top"].set_visible(False); ax_f.spines["right"].set_visible(False)

plt.suptitle("30-Day Hospital Readmission Analysis Report\nFigure 1: Key Results Summary",
             fontsize=13,y=1.01)
plt.savefig("/tmp/mod08/quarto_figure1.png",bbox_inches="tight",dpi=300)
plt.show()
print(f"Publication figure saved at 300 dpi | AUC={auc:.3f}")


## 5. Programmatic Table 1

In [ ]:
# Generate publication-ready Table 1 (patient characteristics)
from scipy.stats import chi2_contingency, mannwhitneyu

def table1_row_continuous(df_all, col, group_col="readmitted", label=None):
    label = label or col
    overall = f"{df_all[col].mean():.1f} ({df_all[col].std():.1f})"
    g0 = df_all[df_all[group_col]==0][col]
    g1 = df_all[df_all[group_col]==1][col]
    stat, p = mannwhitneyu(g0,g1,alternative="two-sided")
    return {"Variable":label,"Overall":overall,
            "Not Readmitted":f"{g0.mean():.1f} ({g0.std():.1f})",
            "Readmitted":f"{g1.mean():.1f} ({g1.std():.1f})",
            "p-value":f"{p:.3f}"}

def table1_row_binary(df_all, col, group_col="readmitted", label=None):
    label = label or col
    overall = f"{df_all[col].mean()*100:.1f}%"
    g0 = df_all[df_all[group_col]==0][col]
    g1 = df_all[df_all[group_col]==1][col]
    ct = pd.crosstab(df_all[col], df_all[group_col])
    _, p, _, _ = chi2_contingency(ct)
    return {"Variable":label,"Overall":overall,
            "Not Readmitted":f"{g0.mean()*100:.1f}%",
            "Readmitted":f"{g1.mean()*100:.1f}%",
            "p-value":f"{p:.3f}"}

rows = [
    {"Variable":"N","Overall":str(len(df)),
     "Not Readmitted":str((df.readmitted==0).sum()),
     "Readmitted":str((df.readmitted==1).sum()),"p-value":""},
    table1_row_continuous(df,"age","readmitted","Age (years), mean (SD)"),
    table1_row_continuous(df,"los_days","readmitted","Length of stay (days), mean (SD)"),
    table1_row_binary(df,"diabetes","readmitted","Diabetes, n (%)"),
    table1_row_binary(df,"hf","readmitted","Heart failure, n (%)"),
    table1_row_binary(df,"smoking","readmitted","Current smoker, n (%)"),
]
table1 = pd.DataFrame(rows)
print("Table 1: Patient Characteristics by Readmission Status")
print("="*75)
display(table1)

# Save as CSV for Quarto/Word
table1.to_csv("/tmp/mod08/table1.csv",index=False)
print("\nTable 1 saved to /tmp/mod08/table1.csv")


## 6. Quarto Rendering Commands

In [ ]:
# Quarto rendering commands and cross-referencing examples
QUARTO_COMMANDS = (
    "# Install Quarto (once):\n"
    "# wget https://quarto.org/download/latest/quarto-linux-amd64.deb\n"
    "# sudo dpkg -i quarto-linux-amd64.deb\n\n"
    "# Render to HTML (self-contained):\n"
    "# quarto render report.qmd --to html\n\n"
    "# Render to PDF (requires LaTeX):\n"
    "# quarto render report.qmd --to pdf\n\n"
    "# Render to Word:\n"
    "# quarto render report.qmd --to docx\n\n"
    "# Render all formats:\n"
    "# quarto render report.qmd\n\n"
    "# Preview with live reload:\n"
    "# quarto preview report.qmd\n\n"
    "# Publish to Quarto Pub:\n"
    "# quarto publish quarto-pub report.qmd\n\n"
    "# Cross-references in .qmd:\n"
    "# @fig-roc-curve   -> Figure 1\n"
    "# @tbl-table1      -> Table 1\n"
    "# @sec-methods     -> Section 2\n"
    "# @eq-logistic     -> Equation 1\n"
)
print(QUARTO_COMMANDS)


## 7. Advanced Quarto Features

In [ ]:
# Quarto panels, tabs, and callouts
QUARTO_ADVANCED = (
    "## Panel Layout\n"
    "::: {layout-ncol=2}\n"
    "```{python}\n"
    "#| label: fig-roc\n"
    "plot_roc()\n"
    "```\n"
    "```{python}\n"
    "#| label: fig-calibration\n"
    "plot_calibration()\n"
    "```\n"
    ":::\n\n"
    "## Tabset\n"
    "::: {.panel-tabset}\n"
    "### Training Set\n"
    "Content for training set...\n"
    "### Test Set\n"
    "Content for test set...\n"
    ":::\n\n"
    "## Callout Boxes\n"
    "::: {.callout-warning}\n"
    "### Model Limitations\n"
    "This model was trained on a single-site cohort and may not generalise.\n"
    ":::\n\n"
    "::: {.callout-tip}\n"
    "### Clinical Application\n"
    "Flag patients with predicted probability > 0.30 for care transition support.\n"
    ":::\n\n"
    "## Conditional Content (for-loop tables)\n"
    "```{python}\n"
    "#| output: asis\n"
    "for payer in df.payer.unique():\n"
    "    sub = df[df.payer==payer]\n"
    "    print(f'### {payer} (n={len(sub)})')\n"
    "    display(sub.describe())\n"
    "```\n"
)
print(QUARTO_ADVANCED)


## 8. Knowledge Check
1. What is the advantage of Quarto's `execute: cache: true` for a long-running model?
2. How do cross-references (`@fig-roc`) differ from manually typing "Figure 1"?
3. A journal requires a Word `.docx` with specific formatting. How does Quarto handle this?
4. Your `.qmd` renders fine locally but fails in CI because `quarto` is not installed. How do you fix this?
5. Build a Quarto parameterised report that generates separate HTML pages for each hospital unit.


In [ ]:
# Exercise 5 — Parameterised Quarto report
PARAMETERISED_QMD = (
    "---\n"
    "title: 'Unit Report: {{< params.unit_name >}}'\n"
    "params:\n"
    "  unit_name: 'Cardiology'\n"
    "  min_date:  '2023-01-01'\n"
    "  max_date:  '2023-12-31'\n"
    "---\n\n"
    "```{python}\n"
    "unit = params.unit_name\n"
    "df_unit = df[df.unit == unit].copy()\n"
    "print(f'Report for {unit}: N={len(df_unit)} patients')\n"
    "```\n\n"
    "# Render for each unit from command line:\n"
    "# for unit in Cardiology Pulmonology Neurology; do\n"
    "#   quarto render unit_report.qmd -P unit_name:$unit --output ${unit}_report.html\n"
    "# done\n"
)
print("Parameterised report template:")
print(PARAMETERISED_QMD)

# Simulate rendering for multiple units
units = ["Cardiology","Pulmonology","Neurology"]
np.random.seed(42)
df["unit"] = np.random.choice(units, len(df), p=[0.4,0.35,0.25])
for unit in units:
    sub = df[df.unit==unit]
    readmit_r = sub.readmitted.mean()*100
    print(f"Unit: {unit:15s} | N={len(sub):4d} | Readmission rate: {readmit_r:.1f}%")


***
**Next → NB-03: Streamlit Dashboards for Clinical Analytics**
